In [ ]:
!pip install onnx onnxruntime onnx-tool numpy pandas -q

In [ ]:
import os, sys, json, zipfile, math, re
from pathlib import Path
import numpy as np
import onnx
import onnxruntime as ort
print('ENSEMBLE v140 - BLIND LOAD + BEST PICK')

sys.path.insert(0, '/kaggle/input/competitions/neurogolf-2026/neurogolf_utils')
import neurogolf_utils as nu
ort.set_default_logger_severity(3)

OUTPUT_DIR = Path('/kaggle/working')
SUB_DIR = OUTPUT_DIR / 'submission'
SUB_DIR.mkdir(parents=True, exist_ok=True)
print('Config done')


In [ ]:
# ALL SOURCES - no priority, just list them
ALL_SOURCES = [
    # NOTEBOOKS
    Path('/kaggle/input/notebooks/afr1ste/neurogolf-6225-51-public-score-open-solution/submission.zip'),
    Path('/kaggle/input/notebooks/jonathanchan/ngc26-constraint-smart-logic-mix-blending/submission.zip'),
    Path('/kaggle/input/notebooks/konbu17/neurogolf-2026-blended-341-tasks-lb-4215/submission.zip'),
    Path('/kaggle/input/notebooks/magmacot/4200-v5-neurogolf-fix-for-new-system-soon/submission.zip'),
    # DATASETS
    Path('/kaggle/input/datasets/svanikkolli/secret-dataset'),
    Path('/kaggle/input/datasets/svanikkolli/5550-dataset'),
    Path('/kaggle/input/datasets/svanikkolli/lb-4995'),
    Path('/kaggle/input/datasets/svanikkolli/ngc26-dataset'),
    Path('/kaggle/input/datasets/svanikkolli/345-onnx-submission-dataset'),
    Path('/kaggle/input/datasets/svanikkolli/arc-dataset'),
    Path('/kaggle/input/datasets/svanikkolli/artem-nazemtsev-datasets'),
    Path('/kaggle/input/datasets/svanikkolli/boost-dataset'),
    Path('/kaggle/input/datasets/svanikkolli/constraint-smart-logic-ensemble'),
    Path('/kaggle/input/datasets/svanikkolli/cross-dataset'),
    Path('/kaggle/input/datasets/svanikkolli/geometric-dataset'),
    Path('/kaggle/input/datasets/sigmaborov/golf-solve-agent'),
    Path('/kaggle/input/datasets/svanikkolli/infinitesimals-dataset'),
    Path('/kaggle/input/datasets/svanikkolli/jiwei-liu-dataset'),
    Path('/kaggle/input/datasets/svanikkolli/logic-dataset'),
    Path('/kaggle/input/datasets/svanikkolli/magmucot-dataset'),
    Path('/kaggle/input/datasets/svanikkolli/mefa-agi-ensemble-dataset'),
    Path('/kaggle/input/datasets/svanikkolli/neurogolf-2026-starter'),
    Path('/kaggle/input/datasets/yash9439/neurogolf-submission-v1'),
    Path('/kaggle/input/datasets/konbu17/neurogolf-2026-blended-341'),
    Path('/kaggle/input/datasets/svanikkolli/nnmax-dataset'),
    Path('/kaggle/input/datasets/svanikkolli/rule-based-dataset'),
    Path('/kaggle/input/datasets/svanikkolli/saxaphone-dataset'),
    Path('/kaggle/input/datasets/svanikkolli/scorer-arc-appx'),
    Path('/kaggle/input/datasets/svanikkolli/tiny-onnx-dataset'),
    Path('/kaggle/input/datasets/svanikkolli/triage-system-dataset'),
    Path('/kaggle/input/datasets/svanikkolli/watermelon-2-dataset'),
    Path('/kaggle/input/datasets/svanikkolli/watermelon-dataset'),
    Path('/kaggle/input/datasets/svanikkolli/yash-dataset'),
    Path('/kaggle/input/datasets/karnakbaevarthur/logic-for-each-arc-task'),
    Path('/kaggle/input/datasets/konbu17/neurogolf-2026-blended-401-v117'),
]


In [ ]:
def load_any(path):
    models = {}
    if not path.exists(): return models
    try:
        # Try ZIP
        if path.is_file() and path.suffix.lower() == '.zip':
            with zipfile.ZipFile(path, 'r') as zf:
                for entry in zf.namelist():
                    m = re.match(r'task(\d{3})\.onnx', os.path.basename(entry))
                    if m: models[int(m.group(1))] = zf.read(entry)
            return models
        # Try directory
        for f in path.glob('*.zip'):
            with zipfile.ZipFile(f, 'r') as z:
                for name in z.namelist():
                    m = re.match(r'task(\d{3})\.onnx', os.path.basename(name))
                    if m: models[int(m.group(1))] = z.read(name)
            if models: return models
        for f in path.glob('*.onnx'):
            m = re.match(r'task(\d{3})\.onnx', f.name)
            if m: models[int(m.group(1))] = f.read_bytes()
    except: pass
    return models

print('BLIND LOADING everything...')
all_sources = {}
for path in ALL_SOURCES:
    label = path.name if path.is_dir() else path.stem
    models = load_any(path)
    if models: all_sources[label] = models
    print(f'  [{label}] {len(models)}')

print(f'TOTAL: {sum(len(m) for m in all_sources.values())} models from {len(all_sources)} sources')


In [ ]:
def validate_official(task_id, raw):
    try:
        sess = ort.InferenceSession(raw, providers=['CPUExecutionProvider'])
    except: return None
    try:
        examples = nu.load_examples(task_id)
    except: return None
    try:
        agi_pass, agi_fail, _ = nu.verify_subset(sess, examples['train'] + examples['test'])
        gen_pass, gen_fail, _ = nu.verify_subset(sess, examples['arc-gen'])
    except: return None
    if agi_fail > 0 or gen_fail > 0: return None
    try:
        tmp = f'/tmp/v{task_id:03d}.onnx'
        with open(tmp, 'wb') as f: f.write(raw)
        macs, mem, params = nu.score_network(tmp)
        if None in (macs, mem, params): return None
        cost = macs + mem + params
    except: return None
    return {'cost': cost, 'score': max(1.0, 25.0 - math.log(cost))}


In [ ]:
print('=== ENSEMBLE v140 - COMPARE ALL THEN BEST ===')

solved = {}
results = []

for task_id in range(1, 401):
    # Collect ALL candidates from ALL sources for this task
    candidates = []
    for src, models in all_sources.items():
        if task_id not in models: continue
        raw = models[task_id]
        r = validate_official(task_id, raw)
        if r: 
            # Store: (cost, score, raw, source_name)
            candidates.append((r['cost'], r['score'], raw, src))
    
    if candidates:
        # BLIND COMPARE: Sort by (lowest cost, then highest score)
        candidates.sort(key=lambda x: (x[0], -x[1]))
        # Pick best
        cost, score, raw, src = candidates[0]
        solved[task_id] = raw
        results.append({'task': task_id, 'cost': cost, 'score': score, 'source': src})
    
    if task_id % 50 == 0: 
        print(f'  [{task_id}] solved={len(solved)}')

print(f'\nSolved: {len(solved)}')


In [ ]:
import pandas as pd
df = pd.DataFrame(results)
print('=== Results ===')
print(f'Solved: {len(solved)}/400')
print(f'Score: {df["score"].sum():.2f}')
print()
print('Top sources that won:')
print(df['source'].value_counts())


In [ ]:
print('=== Saving ===')
for tid, raw in solved.items():
    (SUB_DIR / f'task{tid:03d}.onnx').write_bytes(raw)
zip_path = OUTPUT_DIR / 'submission.zip'
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(SUB_DIR.glob('*.onnx')): zf.write(f, f.name)
print(f'Wrote {len(solved)} files')
